# Belka · 03 · Ouro
Dado pronto para uso: um modelo dimensional (estrela) com o **nível de risco por município e por hora**,
que é o que o MVP mostra ao cidadão.

| Tabela | Tipo | Granularidade | Chave |
|---|---|---|---|
| `ouro_fato_risco_hora` | fato | 1 linha por município por hora | `municipio_id` + `tempo_id` |
| `ouro_dim_municipio` | dimensão | 1 linha por município (com população do IBGE) | `municipio_id` |
| `ouro_dim_tempo` | dimensão | 1 linha por hora do período | `tempo_id` (AAAAMMDDHH) |
| `ouro_dim_nivel_risco` | dimensão | 1 linha por nível da Belka | `nivel_id` (0 a 3) |

**Por que recriar em vez de MERGE:** a Ouro é derivada da Prata e as somas de chuva em 24 h e 72 h
dependem das horas anteriores. Recalcular tudo a cada execução (`CREATE OR REPLACE`, atômico no Delta)
garante que nenhum nível fique desatualizado e que rodar de novo dê sempre o mesmo resultado.

In [0]:
%sql
-- Mesmo fuso da Prata: as horas são locais de São Paulo.
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## Limiares dos níveis (versão v1)
Os níveis são **indicadores experimentais** calculados por limiares documentados, **não são alerta oficial**:
a Belka complementa a Defesa Civil. Os limiares seguem a lógica dos avisos do INMET (perigo potencial, perigo,
grande perigo), que a escala de cores da Belka já espelha.

| Risco | Sinal usado | Fique de olho (1) | Prepare-se (2) | Aja agora (3) |
|---|---|---|---|---|
| Enchente | chuva na hora **ou** acumulada em 24 h | ≥ 20 mm/h ou ≥ 30 mm | ≥ 30 mm/h ou ≥ 50 mm | ≥ 60 mm/h ou ≥ 100 mm |
| Deslizamento | chuva acumulada em 72 h (solo encharcado) | ≥ 50 mm | ≥ 80 mm | ≥ 120 mm |
| Calor extremo | temperatura | ≥ 32 °C | ≥ 35 °C | ≥ 38 °C |
| Tempo seco (incêndio) | umidade relativa | ≤ 30 % | ≤ 20 % | < 12 % |
| Vento forte | rajada | ≥ 40 km/h | ≥ 60 km/h | ≥ 100 km/h |

O nível da hora é o **maior** entre os cinco riscos, e o risco que o define vira `risco_principal`, com um
`motivo` em linguagem simples (a explicação de por que alertou). Umidade ou rajada nulas não geram nível.
As somas de 24 h e 72 h usam as horas que existem: nas primeiras horas do período a janela é parcial.

In [0]:
%sql
CREATE OR REPLACE TEMPORARY FUNCTION belka_nivel_enchente(chuva_hora DOUBLE, chuva_24h DOUBLE)
RETURNS INT
RETURN CASE
  WHEN chuva_hora >= 60 OR chuva_24h >= 100 THEN 3
  WHEN chuva_hora >= 30 OR chuva_24h >= 50  THEN 2
  WHEN chuva_hora >= 20 OR chuva_24h >= 30  THEN 1
  ELSE 0 END;

CREATE OR REPLACE TEMPORARY FUNCTION belka_nivel_deslizamento(chuva_72h DOUBLE)
RETURNS INT
RETURN CASE
  WHEN chuva_72h >= 120 THEN 3
  WHEN chuva_72h >= 80  THEN 2
  WHEN chuva_72h >= 50  THEN 1
  ELSE 0 END;

CREATE OR REPLACE TEMPORARY FUNCTION belka_nivel_calor(temperatura DOUBLE)
RETURNS INT
RETURN CASE
  WHEN temperatura >= 38 THEN 3
  WHEN temperatura >= 35 THEN 2
  WHEN temperatura >= 32 THEN 1
  ELSE 0 END;

CREATE OR REPLACE TEMPORARY FUNCTION belka_nivel_tempo_seco(umidade INT)
RETURNS INT
RETURN CASE
  WHEN umidade < 12  THEN 3
  WHEN umidade <= 20 THEN 2
  WHEN umidade <= 30 THEN 1
  ELSE 0 END;

CREATE OR REPLACE TEMPORARY FUNCTION belka_nivel_vento(rajada DOUBLE)
RETURNS INT
RETURN CASE
  WHEN rajada >= 100 THEN 3
  WHEN rajada >= 60  THEN 2
  WHEN rajada >= 40  THEN 1
  ELSE 0 END;

-- Número com uma casa e vírgula decimal, para o texto do motivo
CREATE OR REPLACE TEMPORARY FUNCTION belka_fmt(x DOUBLE)
RETURNS STRING
RETURN replace(format_string('%.1f', x), '.', ',');

## Dimensão nível de risco
Os níveis do manual de marca da Belka. Cor nunca vai sozinha: cada nível tem nome, ícone, atitude e orientação,
e a cor do texto garante contraste (WCAG). O nível 0 existe para a tela mostrar “sem alerta” em vez de vazio.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.ouro_dim_nivel_risco
COMMENT 'Belka · níveis de risco (manual de marca). Indicador experimental, não é alerta oficial.'
AS SELECT * FROM VALUES
  (0, 'Sem alerta',    'check', '#ECE7DE', '#1E2635', 'Siga a rotina',
      'Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.',                 'sem aviso',        false),
  (1, 'Fique de olho', 'eye',   '#F5C518', '#0B1B34', 'Acompanhe',
      'Acompanhe as atualizações e evite riscos desnecessários.',                         'perigo potencial', true),
  (2, 'Prepare-se',    'backpack', '#F07C1B', '#0B1B34', 'Prepare-se',
      'Separe documentos, remédios e água. Combine com a família onde se encontrar.',     'perigo',           true),
  (3, 'Aja agora',     'siren', '#C81E35', '#FFFFFF', 'Aja agora',
      'Saia da área de risco e vá para o ponto de apoio mais próximo.',                   'grande perigo',    true)
AS t(nivel_id, nivel_nome, icone, cor_fundo_hex, cor_texto_hex, atitude, orientacao, equivalente_inmet, e_alerta);

num_affected_rows,num_inserted_rows


## Dimensão tempo
Uma linha para **cada hora** entre a primeira e a última hora da Prata, gerada com `sequence`, para que
uma hora sem dado apareça como buraco nas consultas em vez de sumir. `tempo_id` = AAAAMMDDHH.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.ouro_dim_tempo
COMMENT 'Belka · calendário horário do período coletado (hora local America/Sao_Paulo)'
AS
WITH limites AS (
  SELECT MIN(data_hora) AS inicio, MAX(data_hora) AS fim
  FROM workspace.belka.prata_clima_hora
),
horas AS (
  SELECT explode(sequence(inicio, fim, INTERVAL 1 HOUR)) AS data_hora FROM limites
)
SELECT
  CAST(date_format(data_hora, 'yyyyMMddHH') AS BIGINT) AS tempo_id,
  data_hora,
  to_date(data_hora)    AS data,
  year(data_hora)       AS ano,
  month(data_hora)      AS mes,
  day(data_hora)        AS dia,
  hour(data_hora)       AS hora,
  dayofweek(data_hora)  AS dia_semana_num,
  CASE dayofweek(data_hora)
    WHEN 1 THEN 'domingo' WHEN 2 THEN 'segunda' WHEN 3 THEN 'terça' WHEN 4 THEN 'quarta'
    WHEN 5 THEN 'quinta'  WHEN 6 THEN 'sexta'   ELSE 'sábado' END AS dia_semana,
  dayofweek(data_hora) IN (1, 7) AS fim_de_semana,
  CASE
    WHEN hour(data_hora) < 6  THEN 'madrugada'
    WHEN hour(data_hora) < 12 THEN 'manhã'
    WHEN hour(data_hora) < 18 THEN 'tarde'
    ELSE 'noite' END AS periodo_dia
FROM horas;

num_affected_rows,num_inserted_rows


## Dimensão município
Junta as duas fontes: coordenadas vêm do Open-Meteo (Prata clima) e a população do IBGE (Prata população,
ano mais recente). `contexto_territorio` é um texto descritivo curto, escrito pela equipe, para a tela.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.ouro_dim_municipio
COMMENT 'Belka · municípios piloto com população estimada do IBGE'
AS
WITH clima AS (
  SELECT municipio_id,
         max_by(municipio_nome, data_hora) AS municipio_nome,
         max_by(uf, data_hora)             AS uf,
         max_by(latitude, data_hora)       AS latitude,
         max_by(longitude, data_hora)      AS longitude
  FROM workspace.belka.prata_clima_hora
  GROUP BY municipio_id
),
pop AS (
  SELECT municipio_id, municipio_nome, uf, ano, populacao
  FROM workspace.belka.prata_populacao
  QUALIFY row_number() OVER (PARTITION BY municipio_id ORDER BY ano DESC) = 1
)
SELECT
  c.municipio_id,
  coalesce(p.municipio_nome, c.municipio_nome) AS municipio_nome,
  coalesce(p.uf, c.uf)                         AS uf,
  c.latitude,
  c.longitude,
  p.populacao,
  p.ano AS ano_populacao,
  CASE c.municipio_id
    WHEN 3550308 THEN 'Capital: córregos canalizados e várzeas sujeitas a alagamento'
    WHEN 3518800 THEN 'Região metropolitana: várzea do Tietê e encostas na periferia'
    WHEN 3516408 THEN 'Região metropolitana: rio Juqueri e encostas com histórico de deslizamento'
    WHEN 3509502 THEN 'Interior: calor e tempo seco no inverno e na primavera'
    WHEN 3550704 THEN 'Litoral norte: encostas da Serra do Mar com histórico de deslizamento'
  END AS contexto_territorio
FROM clima c
LEFT JOIN pop p ON p.municipio_id = c.municipio_id;

num_affected_rows,num_inserted_rows


## Fato: risco por município e hora
1. Soma a chuva das últimas 24 h e 72 h de cada município (janela pelo relógio, não pela posição da linha).
2. Calcula o nível de cada um dos cinco riscos com as funções acima.
3. O nível da hora é o maior deles; o risco que o define vira `risco_principal` e ganha um `motivo` legível.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.belka.ouro_fato_risco_hora
COMMENT 'Belka · nível de risco experimental por município e hora (regras v1). Fonte: prata_clima_hora.'
AS
WITH base AS (
  SELECT
    municipio_id, data_hora, temperatura_c, precipitacao_mm, umidade_pct, rajada_kmh, coletado_em,
    SUM(precipitacao_mm) OVER (PARTITION BY municipio_id ORDER BY unix_timestamp(data_hora)
                               RANGE BETWEEN 82800 PRECEDING AND CURRENT ROW)  AS chuva_24h_mm,  -- 23 h antes + a hora
    SUM(precipitacao_mm) OVER (PARTITION BY municipio_id ORDER BY unix_timestamp(data_hora)
                               RANGE BETWEEN 255600 PRECEDING AND CURRENT ROW) AS chuva_72h_mm   -- 71 h antes + a hora
  FROM workspace.belka.prata_clima_hora
),
niveis AS (
  SELECT *,
    belka_nivel_enchente(precipitacao_mm, chuva_24h_mm) AS nivel_enchente,
    belka_nivel_deslizamento(chuva_72h_mm)              AS nivel_deslizamento,
    belka_nivel_calor(temperatura_c)                    AS nivel_calor,
    belka_nivel_tempo_seco(umidade_pct)                 AS nivel_tempo_seco,
    belka_nivel_vento(rajada_kmh)                       AS nivel_vento
  FROM base
),
classificado AS (
  SELECT *,
    greatest(nivel_enchente, nivel_deslizamento, nivel_calor, nivel_tempo_seco, nivel_vento) AS nivel_id
  FROM niveis
),
com_risco AS (
  SELECT *,
    CASE
      WHEN nivel_id = 0                  THEN 'nenhum'
      WHEN nivel_deslizamento = nivel_id THEN 'deslizamento'
      WHEN nivel_enchente     = nivel_id THEN 'enchente'
      WHEN nivel_calor        = nivel_id THEN 'calor extremo'
      WHEN nivel_tempo_seco   = nivel_id THEN 'tempo seco'
      ELSE 'vento forte' END AS risco_principal
  FROM classificado
)
SELECT
  municipio_id,
  CAST(date_format(data_hora, 'yyyyMMddHH') AS BIGINT) AS tempo_id,
  nivel_id,
  data_hora,
  temperatura_c,
  precipitacao_mm,
  round(chuva_24h_mm, 1) AS chuva_24h_mm,
  round(chuva_72h_mm, 1) AS chuva_72h_mm,
  umidade_pct,
  rajada_kmh,
  nivel_enchente, nivel_deslizamento, nivel_calor, nivel_tempo_seco, nivel_vento,
  risco_principal,
  CASE risco_principal
    WHEN 'enchente'      THEN concat('Chuva de ', belka_fmt(precipitacao_mm), ' mm na hora e ',
                                     belka_fmt(chuva_24h_mm), ' mm em 24 h')
    WHEN 'deslizamento'  THEN concat(belka_fmt(chuva_72h_mm), ' mm de chuva acumulada em 72 h')
    WHEN 'calor extremo' THEN concat('Temperatura de ', belka_fmt(temperatura_c), ' °C')
    WHEN 'tempo seco'    THEN concat('Umidade do ar em ', umidade_pct, ' %')
    WHEN 'vento forte'   THEN concat('Rajadas de ', belka_fmt(rajada_kmh), ' km/h')
    ELSE 'Nenhum sinal acima do limiar'
  END AS motivo,
  'v1' AS versao_regras,
  coletado_em,
  current_timestamp() AS atualizado_em
FROM com_risco;

num_affected_rows,num_inserted_rows


## Conferência da Ouro
Esperado: fato 1.560 linhas (5 municípios × 13 dias × 24 h), dimensão tempo 312, municípios 5, níveis 4.

In [0]:
%sql
SELECT 'ouro_fato_risco_hora' AS tabela, COUNT(*) AS linhas FROM workspace.belka.ouro_fato_risco_hora
UNION ALL SELECT 'ouro_dim_municipio',   COUNT(*) FROM workspace.belka.ouro_dim_municipio
UNION ALL SELECT 'ouro_dim_tempo',       COUNT(*) FROM workspace.belka.ouro_dim_tempo
UNION ALL SELECT 'ouro_dim_nivel_risco', COUNT(*) FROM workspace.belka.ouro_dim_nivel_risco;

tabela,linhas
ouro_fato_risco_hora,1560
ouro_dim_municipio,5
ouro_dim_tempo,312
ouro_dim_nivel_risco,4


In [0]:
%sql
-- Integridade do modelo: as três contagens devem ser 0
SELECT
  (SELECT COUNT(*) FROM (SELECT municipio_id, tempo_id FROM workspace.belka.ouro_fato_risco_hora
                         GROUP BY ALL HAVING COUNT(*) > 1))                              AS chaves_duplicadas,
  (SELECT COUNT(*) FROM workspace.belka.ouro_fato_risco_hora f
     LEFT ANTI JOIN workspace.belka.ouro_dim_municipio m ON f.municipio_id = m.municipio_id) AS sem_municipio,
  (SELECT COUNT(*) FROM workspace.belka.ouro_fato_risco_hora f
     LEFT ANTI JOIN workspace.belka.ouro_dim_tempo t ON f.tempo_id = t.tempo_id)            AS sem_tempo,
  (SELECT COUNT(*) FROM workspace.belka.ouro_fato_risco_hora f
     LEFT ANTI JOIN workspace.belka.ouro_dim_nivel_risco n ON f.nivel_id = n.nivel_id)      AS sem_nivel;

chaves_duplicadas,sem_municipio,sem_tempo,sem_nivel
0,0,0,0


In [0]:
%sql
-- Distribuição geral dos níveis no período
SELECT n.nivel_id, n.nivel_nome, COUNT(f.nivel_id) AS horas
FROM workspace.belka.ouro_dim_nivel_risco n
LEFT JOIN workspace.belka.ouro_fato_risco_hora f ON f.nivel_id = n.nivel_id
GROUP BY n.nivel_id, n.nivel_nome
ORDER BY n.nivel_id;

nivel_id,nivel_nome,horas
0,Sem alerta,1415
1,Fique de olho,145
2,Prepare-se,0
3,Aja agora,0


In [0]:
%sql
SELECT municipio_id, municipio_nome, uf, populacao, ano_populacao, latitude, longitude, contexto_territorio
FROM workspace.belka.ouro_dim_municipio
ORDER BY populacao DESC;

municipio_id,municipio_nome,uf,populacao,ano_populacao,latitude,longitude,contexto_territorio
3550308,São Paulo,SP,11911337,2026,-23.514938,-46.610504,Capital: córregos canalizados e várzeas sujeitas a alagamento
3518800,Guarulhos,SP,1351284,2026,-23.44464,-46.561523,Região metropolitana: várzea do Tietê e encostas na periferia
3509502,Campinas,SP,1189761,2026,-22.952549,-47.066803,Interior: calor e tempo seco no inverno e na primavera
3516408,Franco da Rocha,SP,150648,2026,-23.304043,-46.74713,Região metropolitana: rio Juqueri e encostas com histórico de deslizamento
3550704,São Sebastião,SP,84514,2026,-23.725836,-45.427673,Litoral norte: encostas da Serra do Mar com histórico de deslizamento


## Teste dos limiares
Num período curto é normal não haver horas em Prepare-se ou Aja agora. Este teste passa valores inventados
(só em memória) pelas **mesmas funções** e mostra que cada nível é atingido onde deve.

In [0]:
%sql
SELECT caso,
       belka_nivel_enchente(chuva_hora, chuva_24h) AS enchente,
       belka_nivel_deslizamento(chuva_72h)          AS deslizamento,
       belka_nivel_calor(temperatura)               AS calor,
       belka_nivel_tempo_seco(umidade)              AS tempo_seco,
       belka_nivel_vento(rajada)                    AS vento
FROM VALUES
  ('dia comum',                  0.0,   0.0,   0.0, 24.0, 70,   15.0),
  ('pancada de 25 mm/h',        25.0,  25.0,  25.0, 24.0, 90,   20.0),
  ('55 mm em 24 h',              8.0,  55.0,  60.0, 22.0, 95,   25.0),
  ('temporal de 70 mm/h',       70.0,  90.0,  90.0, 21.0, 98,   65.0),
  ('130 mm em 72 h',             2.0,  40.0, 130.0, 20.0, 97,   20.0),
  ('tarde de 36 °C',             0.0,   0.0,   0.0, 36.0, 28,   20.0),
  ('39 °C e umidade de 10 %',    0.0,   0.0,   0.0, 39.0, 10,   30.0),
  ('umidade e rajada nulas',     0.0,   0.0,   0.0, 24.0, NULL, NULL)
  AS t(caso, chuva_hora, chuva_24h, chuva_72h, temperatura, umidade, rajada);

caso,enchente,deslizamento,calor,tempo_seco,vento
dia comum,0,0,0,0,0
pancada de 25 mm/h,1,0,0,0,0
55 mm em 24 h,2,1,0,0,0
temporal de 70 mm/h,3,2,0,0,2
130 mm em 72 h,1,3,0,0,0
tarde de 36 °C,0,0,2,1,0
39 °C e umidade de 10 %,0,0,3,3,0
umidade e rajada nulas,0,0,0,0,0


# As 5 consultas do MVP
Cada consulta responde a uma pergunta que a tela da Belka faz pelo cidadão ou pela Defesa Civil.
A **consulta 5 é a de impacto**: quantas pessoas vivem onde o nível chegou a Prepare-se ou Aja agora.

### Consulta 1 · “Como está o meu município hoje?”
Tela inicial do app: um cartão por município com o **pior nível do último dia coletado**, o risco, o motivo,
a hora crítica, a cor da escala e a orientação. É o que o cidadão vê ao abrir a Belka.

In [0]:
%sql
WITH ultimo_dia AS (
  SELECT MAX(t.data) AS data
  FROM workspace.belka.ouro_fato_risco_hora f
  JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
),
pior_hora AS (
  SELECT f.municipio_id, f.nivel_id, f.risco_principal, f.motivo, t.data_hora, t.data,
         MAX(f.temperatura_c)   OVER (PARTITION BY f.municipio_id) AS temp_max_c,
         MIN(f.umidade_pct)     OVER (PARTITION BY f.municipio_id) AS umidade_min_pct,
         SUM(f.precipitacao_mm) OVER (PARTITION BY f.municipio_id) AS chuva_dia_mm
  FROM workspace.belka.ouro_fato_risco_hora f
  JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
  WHERE t.data = (SELECT data FROM ultimo_dia)
  QUALIFY row_number() OVER (PARTITION BY f.municipio_id ORDER BY f.nivel_id DESC, f.data_hora) = 1
)
SELECT
  date_format(p.data, 'dd/MM/yyyy')  AS dia,
  m.municipio_nome,
  n.nivel_nome,
  n.cor_fundo_hex,
  p.risco_principal,
  p.motivo,
  CASE WHEN p.nivel_id > 0 THEN date_format(p.data_hora, 'HH:mm') END AS hora_critica,
  round(p.temp_max_c, 1)             AS temp_max_c,
  p.umidade_min_pct,
  round(p.chuva_dia_mm, 1)           AS chuva_dia_mm,
  n.orientacao
FROM pior_hora p
JOIN workspace.belka.ouro_dim_municipio   m ON m.municipio_id = p.municipio_id
JOIN workspace.belka.ouro_dim_nivel_risco n ON n.nivel_id     = p.nivel_id
ORDER BY p.nivel_id DESC, m.populacao DESC;

dia,municipio_nome,nivel_nome,cor_fundo_hex,risco_principal,motivo,hora_critica,temp_max_c,umidade_min_pct,chuva_dia_mm,orientacao
06/11/2026,São Paulo,Sem alerta,#ECE7DE,nenhum,Nenhum sinal acima do limiar,null,26.3,67,14.4,Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.
06/11/2026,Guarulhos,Sem alerta,#ECE7DE,nenhum,Nenhum sinal acima do limiar,null,27.8,60,9.9,Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.
06/11/2026,Campinas,Sem alerta,#ECE7DE,nenhum,Nenhum sinal acima do limiar,null,29.5,60,7.3,Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.
06/11/2026,Franco da Rocha,Sem alerta,#ECE7DE,nenhum,Nenhum sinal acima do limiar,null,28.2,60,7.2,Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.
06/11/2026,São Sebastião,Sem alerta,#ECE7DE,nenhum,Nenhum sinal acima do limiar,null,26.0,80,18.7,Nenhum sinal acima do limiar. Acompanhe a previsão normalmente.


### Consulta 2 · “Quais municípios passaram mais tempo em alerta?”
Painel da Defesa Civil: horas em cada nível no período e a porcentagem de horas em alerta (nível ≥ 1).
Mostra onde a atenção foi mais constante, não só o pico.

In [0]:
%sql
WITH horas AS (
  SELECT municipio_id,
         COUNT(*)                AS horas_total,
         COUNT_IF(nivel_id = 0)  AS h_sem_alerta,
         COUNT_IF(nivel_id = 1)  AS h_fique_de_olho,
         COUNT_IF(nivel_id = 2)  AS h_prepare_se,
         COUNT_IF(nivel_id = 3)  AS h_aja_agora,
         COUNT_IF(nivel_id >= 1) AS h_em_alerta,
         MAX(nivel_id)           AS nivel_max
  FROM workspace.belka.ouro_fato_risco_hora
  GROUP BY municipio_id
)
SELECT
  m.municipio_nome,
  h.horas_total, h.h_sem_alerta, h.h_fique_de_olho, h.h_prepare_se, h.h_aja_agora,
  round(100.0 * h.h_em_alerta / h.horas_total, 1) AS pct_horas_em_alerta,
  n.nivel_nome AS nivel_maximo_no_periodo
FROM horas h
JOIN workspace.belka.ouro_dim_municipio   m ON m.municipio_id = h.municipio_id
JOIN workspace.belka.ouro_dim_nivel_risco n ON n.nivel_id     = h.nivel_max
ORDER BY h.h_em_alerta DESC, h.nivel_max DESC;

municipio_nome,horas_total,h_sem_alerta,h_fique_de_olho,h_prepare_se,h_aja_agora,pct_horas_em_alerta,nivel_maximo_no_periodo
São Sebastião,312,266,46,0,0,14.7,Fique de olho
Campinas,312,276,36,0,0,11.5,Fique de olho
São Paulo,312,290,22,0,0,7.1,Fique de olho
Franco da Rocha,312,291,21,0,0,6.7,Fique de olho
Guarulhos,312,292,20,0,0,6.4,Fique de olho


### Consulta 3 · “Qual sinal chegou mais perto do perigo em cada município?”
Tela de detalhe do município: o pior valor de cada sinal no período ao lado do limiar de Fique de olho,
e quantas horas cada risco foi o principal. Explica **por que** a Belka alertou (ou por que ficou em silêncio).

In [0]:
%sql
SELECT
  m.municipio_nome,
  round(MAX(f.temperatura_c), 1) AS temp_max_c,       -- limiar 32 °C
  MIN(f.umidade_pct)             AS umidade_min_pct,   -- limiar 30 %
  MAX(f.chuva_24h_mm)            AS chuva_24h_max_mm,  -- limiar 30 mm
  MAX(f.chuva_72h_mm)            AS chuva_72h_max_mm,  -- limiar 50 mm
  round(MAX(f.rajada_kmh), 1)    AS rajada_max_kmh,    -- limiar 40 km/h
  COUNT_IF(f.risco_principal = 'enchente')      AS h_enchente,
  COUNT_IF(f.risco_principal = 'deslizamento')  AS h_deslizamento,
  COUNT_IF(f.risco_principal = 'calor extremo') AS h_calor,
  COUNT_IF(f.risco_principal = 'tempo seco')    AS h_tempo_seco,
  COUNT_IF(f.risco_principal = 'vento forte')   AS h_vento
FROM workspace.belka.ouro_fato_risco_hora f
JOIN workspace.belka.ouro_dim_municipio m ON m.municipio_id = f.municipio_id
GROUP BY m.municipio_nome
ORDER BY temp_max_c DESC;

municipio_nome,temp_max_c,umidade_min_pct,chuva_24h_max_mm,chuva_72h_max_mm,rajada_max_kmh,h_enchente,h_deslizamento,h_calor,h_tempo_seco,h_vento
São Sebastião,32.6,46,37.6,44.8,58.0,30,0,4,0,12
São Paulo,32.4,33,35.5,38.5,45.4,15,0,2,0,5
Campinas,31.3,43,43.9,44.4,49.0,22,0,0,0,14
Guarulhos,31.1,37,35.6,41.6,48.6,12,0,0,0,8
Franco da Rocha,30.3,43,33.8,35.9,47.2,14,0,0,0,7


### Consulta 4 · “Em que horário do dia o risco é maior?”
Orientação de rotina: em que período do dia os alertas se concentram e como ficam calor, umidade, chuva
e vento em cada período. Vira dica prática no app (ex.: evitar sol forte em certo horário).

In [0]:
%sql
SELECT
  t.periodo_dia,
  COUNT(*)                                          AS horas,
  COUNT_IF(f.nivel_id >= 1)                         AS horas_em_alerta,
  round(100.0 * COUNT_IF(f.nivel_id >= 1) / COUNT(*), 1) AS pct_em_alerta,
  round(AVG(f.temperatura_c), 1)                    AS temp_media_c,
  round(MAX(f.temperatura_c), 1)                    AS temp_max_c,
  round(AVG(f.umidade_pct), 0)                      AS umidade_media_pct,
  round(SUM(f.precipitacao_mm), 1)                  AS chuva_total_mm,
  round(MAX(f.rajada_kmh), 1)                       AS rajada_max_kmh
FROM workspace.belka.ouro_fato_risco_hora f
JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
GROUP BY t.periodo_dia
ORDER BY CASE t.periodo_dia WHEN 'madrugada' THEN 1 WHEN 'manhã' THEN 2 WHEN 'tarde' THEN 3 ELSE 4 END;

periodo_dia,horas,horas_em_alerta,pct_em_alerta,temp_media_c,temp_max_c,umidade_media_pct,chuva_total_mm,rajada_max_kmh
madrugada,390,18,4.6,18.1,23.3,94.0,117.8,47.2
manhã,390,31,7.9,21.0,32.6,82.0,139.2,58.0
tarde,390,58,14.9,23.7,32.5,71.0,156.9,56.5
noite,390,38,9.7,19.8,28.7,89.0,112.5,49.0


### Consulta 5 · IMPACTO · “Quantas pessoas vivem onde o nível chegou a Prepare-se ou Aja agora?”
Para cada dia: municípios cujo pior nível do dia foi Prepare-se ou Aja agora e a **população somada** deles
(IBGE), comparada à população total dos 5 municípios piloto. Também mostra a faixa Fique de olho.
É a medida de alcance da Belka: quantas pessoas teriam recebido um aviso para se preparar ou agir.

In [0]:
%sql
WITH dia AS (
  SELECT f.municipio_id, t.data, MAX(f.nivel_id) AS nivel_max_dia
  FROM workspace.belka.ouro_fato_risco_hora f
  JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
  GROUP BY f.municipio_id, t.data
)
SELECT
  date_format(d.data, 'dd/MM/yyyy') AS dia,
  COUNT_IF(d.nivel_max_dia >= 2)                                          AS municipios_prepare_ou_aja,
  SUM(CASE WHEN d.nivel_max_dia >= 2 THEN coalesce(m.populacao, 0) ELSE 0 END) AS populacao_prepare_ou_aja,
  COUNT_IF(d.nivel_max_dia = 1)                                           AS municipios_fique_de_olho,
  SUM(CASE WHEN d.nivel_max_dia = 1 THEN coalesce(m.populacao, 0) ELSE 0 END)  AS populacao_fique_de_olho,
  SUM(coalesce(m.populacao, 0))                                           AS populacao_piloto,
  round(100.0 * SUM(CASE WHEN d.nivel_max_dia >= 2 THEN coalesce(m.populacao, 0) ELSE 0 END)
        / SUM(coalesce(m.populacao, 0)), 1)                               AS pct_prepare_ou_aja
FROM dia d
JOIN workspace.belka.ouro_dim_municipio m ON m.municipio_id = d.municipio_id
GROUP BY d.data
ORDER BY d.data;

dia,municipios_prepare_ou_aja,populacao_prepare_ou_aja,municipios_fique_de_olho,populacao_fique_de_olho,populacao_piloto,pct_prepare_ou_aja
25/10/2026,0,0,5,14687544,14687544,0.0
26/10/2026,0,0,5,14687544,14687544,0.0
27/10/2026,0,0,3,1424923,14687544,0.0
28/10/2026,0,0,1,1189761,14687544,0.0
29/10/2026,0,0,0,0,14687544,0.0
30/10/2026,0,0,3,13347135,14687544,0.0
31/10/2026,0,0,0,0,14687544,0.0
01/11/2026,0,0,1,84514,14687544,0.0
02/11/2026,0,0,1,1351284,14687544,0.0
03/11/2026,0,0,1,84514,14687544,0.0


In [0]:
%sql
-- Resumo do impacto no período inteiro
WITH dia AS (
  SELECT f.municipio_id, t.data, MAX(f.nivel_id) AS nivel_max_dia
  FROM workspace.belka.ouro_fato_risco_hora f
  JOIN workspace.belka.ouro_dim_tempo t ON t.tempo_id = f.tempo_id
  GROUP BY f.municipio_id, t.data
)
SELECT
  COUNT(DISTINCT d.data)                                                      AS dias_no_periodo,
  COUNT(DISTINCT CASE WHEN d.nivel_max_dia >= 2 THEN d.data END)              AS dias_com_prepare_ou_aja,
  COUNT(DISTINCT CASE WHEN d.nivel_max_dia >= 2 THEN d.municipio_id END)      AS municipios_atingidos,
  SUM(CASE WHEN d.nivel_max_dia >= 2 THEN coalesce(m.populacao, 0) ELSE 0 END) AS pessoas_dia_prepare_ou_aja,
  SUM(CASE WHEN d.nivel_max_dia >= 1 THEN coalesce(m.populacao, 0) ELSE 0 END) AS pessoas_dia_em_algum_alerta
FROM dia d
JOIN workspace.belka.ouro_dim_municipio m ON m.municipio_id = d.municipio_id;

dias_no_periodo,dias_com_prepare_ou_aja,municipios_atingidos,pessoas_dia_prepare_ou_aja,pessoas_dia_em_algum_alerta
13,0,0,0,73456100


## Tabelas da Ouro

In [0]:
%sql
SHOW TABLES IN workspace.belka LIKE 'ouro*';

database,tableName,isTemporary
belka,ouro_dim_municipio,false
belka,ouro_dim_nivel_risco,false
belka,ouro_dim_tempo,false
belka,ouro_fato_risco_hora,false
